# Create an Interpolated ESRGAN Generator Model and Test it on <BR>(i) 300 Image Patches from DIV2K Dataset or <BR> (ii) 14 Images from Set14 Benchmark


Version 1.3

Antonio Esteves @ UMinho, Aug 2024

***

TODO:

* Modify `results_path`
* Modify `datasets_dir`
* Modify `config["gan_pth_path"]`
* Modify `config["psnr_pth_path"]`
* Modify `config["interp_pth_path"]`
* Modify `config["sr_dir"]`
* Modify `config["sr_hr_dir"]`
* Modify `config["lr_dir"]`
* Modify `config["hr_dir"]`
* Modify `config["alpha"]`
* Modify `config["g_rrdb_blocks"]`

In [ ]:
import torch
import os
from   PIL                               import Image
import torch.nn                          as     nn
import torchvision.transforms.functional as     transF
import torch.nn.functional               as     F
from   collections                       import OrderedDict
import matplotlib.pyplot                 as     plt

In [ ]:
USE_INTERPOLATED_GENERATOR = False #### ADJUST THIS VALUE ####

config = {}

config["normalize_images"] = False
results_path               = './results/ESRGAN_Div2k_Flickr2k_05_stage2'
config["gan_pth_path"]     = 'models/ESRGAN_Div2k_Flickr2k_05_gan.pth'
config["psnr_pth_path"]    = 'models/ESRGAN_Div2k_Flickr2k_05_psnr.pth'
config["interp_pth_path"]  = 'models/ESRGAN_Div2k_Flickr2k_05_interpolated.pth'

# ========================= DIV2k-300 Test Dataset =============================

datasets_dir = "OUR_DATASETS_DIR"

config["lr_dir"]           = datasets_dir + '/div2k_300/lr'
config["hr_dir"]           = datasets_dir + '/div2k_300/hr'

if USE_INTERPOLATED_GENERATOR == True:
    #### ADJUST THIS PATH ####
    config["sr_dir"]       = results_path + '/test_div2k_300_interpolated_generator_alpha_0_90/sr'
    #### ADJUST THIS PATH ####
    config["sr_hr_dir"]    = results_path + '/test_div2k_300_interpolated_generator_alpha_0_90/sr_hr'
else:
    config["sr_dir"]       = results_path + '/test_div2k_300_gan_generator/sr'
    config["sr_hr_dir"]    = results_path + '/test_div2k_300_gan_generator/sr_hr'

# ========================= Set14 Test Dataset =================================

#config["lr_dir"]           = datasets_root + '/Set14/LRbicx4'
#config["hr_dir"]           = datasets_root + '/Set14/original'
#
#if USE_INTERPOLATED_GENERATOR == True:
##### ADJUST THIS PATH ####
#    config["sr_dir"]       = results_path + '/test_set14_interpolated_generator_alpha_0_00/sr'
##### ADJUST THIS PATH ####
#    config["sr_hr_dir"]    = results_path + '/test_set14_interpolated_generator_alpha_0_00/sr_hr'
#else:
#    config["sr_dir"]       = results_path + '/test_set14_gan_generator/sr'
#    config["sr_hr_dir"]    = results_path + '/test_set14_gan_generator/sr_hr'

# ==============================================================================

config["alpha"]           = 0.00 #### ADJUST THIS VALUE: 0.00, 0.25, 0.50, 0.75, 0.90 ####
config["channels"]        = 3
config["nf"]              = 64
config["gc"]              = 32
config["scale_factor"]    = 4
config["g_rrdb_blocks"]   = 23 #### ADJUST THIS VALUE: 15 or 23 ####


# Generator Model (copied from training notebook)

In [ ]:
class UpsamplingBlock(nn.Module):
    '''
    Upsampling Block.
    '''
    def __init__(self, nf, scale_factor=4):
        super(UpsamplingBlock, self).__init__()

        self.scale_factor = scale_factor
        if (self.scale_factor == 4):
            self.upsampling1 = nn.Sequential(
                nn.Conv2d(nf, nf, (3, 3), (1, 1), (1, 1)),
                nn.LeakyReLU(0.2, True)
            )
            self.upsampling2 = nn.Sequential(
                nn.Conv2d(nf, nf, (3, 3), (1, 1), (1, 1)),
                nn.LeakyReLU(0.2, True)
            )
        else:
            print('[EROR] Unsupported upsampling factor')
            pass

    def forward(self, x):
        if (self.scale_factor == 4):
            x = self.upsampling1(F.interpolate(x, scale_factor=2, mode="nearest"))
            x = self.upsampling2(F.interpolate(x, scale_factor=2, mode="nearest"))

        return x

In [ ]:
class ResidualDenseBlock(nn.Module):
    '''
    Dense Residual Block.
    It is composed of a sequence of 5 residual blocks.
    Each residual block 'i' (RBi) has a Conv2d, a LeakyReLU, and 
    a skip connections that are used to concatenates the dense block input 'x' 
    with the output from all the previous residual blocks in the sequence  
    RB1_out, ..., RBi_out. The input of each residual block is the result 
    of the concatenation.
    The output of the final residual block is multiplied by 'beta' and added 
    to  input 'x' to produce the Dense Block output.
    '''
    def __init__(self, nf, gc=32, beta=0.2):
        super(ResidualDenseBlock, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 0 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer2 = nn.Sequential(

            nn.Conv2d(
                in_channels  = nf + 1 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1, 
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 2 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 3 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 4 * gc,
                out_channels = nf,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )

        self.beta = beta

    def forward(self, x):
        l1_out = self.layer1(x)
        l2_out = self.layer2(torch.cat((x, l1_out), 1))
        l3_out = self.layer3(torch.cat((x, l1_out, l2_out), 1))
        l4_out = self.layer4(torch.cat((x, l1_out, l2_out, l3_out), 1))
        l5_out = self.layer5(torch.cat((x, l1_out, l2_out, l3_out, l4_out), 1))
        return l5_out.mul(self.beta) + x

In [ ]:
class ResidualInResidualDenseBlock(nn.Module):
    '''
    Residual in Residual Dense Block.

    Sequence of 3 Dense Residual Blocks.
    The output of the final dense residual block is multiplied by 'beta' and added 
    to  input 'x' to produce the RRDB output.
    '''
    def __init__(self, nf, gc=32, beta=0.2):
        super(ResidualInResidualDenseBlock, self).__init__()

        self.layer1 = ResidualDenseBlock(nf, gc)
        self.layer2 = ResidualDenseBlock(nf, gc)
        self.layer3 = ResidualDenseBlock(nf, gc)
        self.beta   = beta

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        return out.mul(self.beta) + x

In [ ]:
class Generator(nn.Module):
    '''
    ESRGAN generator model.

    The generator structure includes:
    * a block with Conv2d->ReLU
    * 'rrdb_blocks'=23 RRDB blocks
    * a block with Conv2d->ReLU
    * an upsample block
    * a block with Conv2d->ReLU
    '''
    def __init__(self, in_channels, out_channels, nf=64, gc=32, scale_factor=4, rrdb_blocks=23):
        super(Generator, self).__init__()

        self.conv1 = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(in_channels, nf, 3), nn.ReLU())

        basic_block_layer = []

        for _ in range(rrdb_blocks):
            basic_block_layer += [ResidualInResidualDenseBlock(nf, gc)]

        self.basic_block = nn.Sequential(*basic_block_layer)

        self.conv2    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, nf, 3), nn.ReLU())
        self.upsample = UpsamplingBlock(nf, scale_factor=scale_factor)
        self.conv3    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, nf, 3), nn.ReLU())
        self.conv4    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, out_channels, 3), nn.ReLU())

    def forward(self, x):
        x1 = self.conv1(x)
        x  = self.basic_block(x1)
        x  = self.conv2(x)
        x  = self.upsample(x + x1)
        x  = self.conv3(x)
        x  = self.conv4(x)
        return x

## Create an Interpolated Generator by Combining the PSNR and GAN Versions of the Trained Generator 

In [ ]:
if not os.path.exists(config["lr_dir"]):
    raise Exception('[INFO] No low-resolution image path')

if not os.path.exists(config["sr_dir"]):
    os.makedirs(config["sr_dir"], exist_ok=True)

if not os.path.exists(config["sr_hr_dir"]):
    os.makedirs(config["sr_hr_dir"], exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = 'cpu'

In [ ]:
if USE_INTERPOLATED_GENERATOR == True:
    generator_PSNR = Generator(
        in_channels    = config["channels"], 
        out_channels   = config["channels"], 
        nf             = config["nf"],
        gc             = config["gc"],
        scale_factor   = config["scale_factor"],
        rrdb_blocks    = config["g_rrdb_blocks"],
    ).to(device)

    generator_GAN = Generator(
        in_channels    = config["channels"], 
        out_channels   = config["channels"], 
        nf             = config["nf"],
        gc             = config["gc"],
        scale_factor   = config["scale_factor"],
        rrdb_blocks    = config["g_rrdb_blocks"],
    ).to(device)

generator_interpolated = Generator(
    in_channels    = config["channels"], 
    out_channels   = config["channels"], 
    nf             = config["nf"],
    gc             = config["gc"],
    scale_factor   = config["scale_factor"],
    rrdb_blocks    = config["g_rrdb_blocks"],
).to(device)

if USE_INTERPOLATED_GENERATOR == True:
    dict_generator_PSNR = torch.load(config["psnr_pth_path"])
    dict_generator_GAN  = torch.load(config["gan_pth_path"])
else:
    dict_generator_GAN  = torch.load(config["gan_pth_path"])

if USE_INTERPOLATED_GENERATOR == True:
    generator_PSNR.load_state_dict(dict_generator_PSNR['generator'])
    generator_GAN.load_state_dict(dict_generator_GAN['generator'])
else:
    generator_interpolated.load_state_dict(dict_generator_GAN['generator'])

if USE_INTERPOLATED_GENERATOR == True:
    state_g_PSNR = generator_PSNR.state_dict()
    state_g_GAN  = generator_GAN.state_dict()

    # Use 'state_g_GAN' to save the interpolation of both generator versions.
    for k, key in enumerate(state_g_PSNR):
        state_g_GAN[key] = state_g_PSNR[key]*(1.0-config["alpha"]) + state_g_GAN[key]*config["alpha"]

    generator_interpolated.load_state_dict(state_g_GAN)

    del generator_PSNR
    del generator_GAN

In [ ]:
def plot_single_SR_HR_image(SR_image, HR_image, file_name, interpolatedG=True, normalize_images=False):
    '''
    Plot and save a single SR image and the corresponding HR image.
    '''

    SR_image = torch.squeeze(SR_image,dim=0)
    HR_image = torch.squeeze(HR_image,dim=0)

    assert len(SR_image.shape) == 3 and len(HR_image.shape) == 3, \
        f'The provided images must have 3 dimensions.'

    sr_cpu    = SR_image.cpu().permute(1,2,0).numpy()
    hr_cpu    = HR_image.cpu().permute(1,2,0).numpy()

    _, ax    = plt.subplots(1, 2, figsize=(2*6,1*6))
    if interpolatedG is True:
        title = 'Super-resolution image generated by the interpolated generator | high-resolution image'
    else:
        title = 'Super-resolution image generated by the GAN generator | high-resolution image'
    plt.suptitle(
        title,
        fontsize   = 15,
        fontweight = 'bold',
    )

    if normalize_images is True:
        ax[0].imshow((sr_cpu+1)/2)
    else:
        ax[0].imshow(sr_cpu)
    ax[0].set_title('generated super-resolution')
    ax[1].imshow(hr_cpu)
    ax[1].set_title('high resolution')

    plt.savefig(file_name, format='png')
    plt.show()
    plt.close()

In [ ]:
from   PIL   import Image
import numpy as     np

def save_single_image(image, file_name, normalize_images=False):
    '''
    Save a single image to file.
    '''

    image = torch.squeeze(image,dim=0)

    assert len(image.shape) == 3 , f'The provided image must have 3 dimensions.'

    img_cpu    = image.cpu().permute(1,2,0).numpy()
    if normalize_images is True:
        img_cpu    = (img_cpu + 1) / 2
    img_cpu   *= 255
    img_cpu    = np.clip(img_cpu, 0,255)

    print(f'[INFO] SR Min={np.min(img_cpu): .1f} Max={np.max(img_cpu): .1f}')

    img_pil    = Image.fromarray(img_cpu.astype('uint8'), mode='RGB')
    img_pil.save(file_name)

## Generate SR Images with the Interpolated Generator

In [ ]:
with torch.no_grad():

    generator_interpolated = generator_interpolated.to(device).eval()

    for image_name in os.listdir(config["lr_dir"]):
        imageLR = Image.open(os.path.join(config["lr_dir"], image_name)).convert('RGB')
        imageLR = transF.to_tensor(imageLR).to(device).unsqueeze(dim=0)
        if config["normalize_images"] is True:
            imageLR = transF.normalize(imageLR, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        imageHR = Image.open(os.path.join(config["hr_dir"], image_name)).convert('RGB')
        imageHR = transF.to_tensor(imageHR).to(device).unsqueeze(dim=0)

        imageSR = generator_interpolated(imageLR)

        fname = os.path.join(config["sr_dir"], image_name)
        save_single_image(imageSR, fname, normalize_images=config["normalize_images"])
        print(f'[INFO] HR Min={255.0*torch.min(imageHR) :.1f} Max={255.0*torch.max(imageHR) :.1f}')
        print(f'[INFO] saved SR image {image_name}')

        fname = os.path.join(config["sr_hr_dir"], image_name)
        plot_single_SR_HR_image(
            imageSR,
            imageHR,
            fname,
            interpolatedG=USE_INTERPOLATED_GENERATOR,
            normalize_images=config["normalize_images"],
        )
        print(f'[INFO] saved SR and HR images {image_name} side-by-side')